# Week 12C: A full-stack Census GIS with a Python backend

Build a working application where a web-map click triggers a live spatial database query. Reuse the Census teaching data from Week 11 and the SQL concepts from 12A.

[Download this notebook](lecture-12C.ipynb) · [Download the complete runnable demo](downloads/census-full-stack.zip) · [Previous: static publishing](lecture-12B.html) · [Project README](examples/census-full-stack/README.md)

**Deliverable:** a local web map that filters historical district counts and reports sample dots within a chosen radius. These synthetic 2010 teaching dots and 2015 boundaries do not describe current demographics or population totals.

## 1. Understand the three layers

| Layer | Technology | Responsibility |
| --- | --- | --- |
| Frontend | HTML, CSS, JavaScript, Leaflet | Display the map, collect category/radius/location, fetch and display results |
| Backend | Python, FastAPI, Uvicorn | Validate requests, run parameterized SQL, return JSON/GeoJSON |
| Database | Persistent DuckDB + Spatial | Store selected Census dots and historical polygons; compute counts and distances |

```mermaid
sequenceDiagram
    participant B as Browser / Leaflet
    participant P as Python / FastAPI
    participant D as DuckDB Spatial
    B->>P: GET /api/nearby?lon=...&lat=...&radius_m=2000
    P->>P: Validate coordinates, radius, category
    P->>D: Parameterized spatial SQL
    D-->>P: Small grouped count table
    P-->>B: JSON response
    B->>B: Update result panel and search circle
```

In 12B the browser loaded already exported files. Here the Python process computes an answer for each request. The frontend and API share one origin (`http://127.0.0.1:8000`), so no separate frontend server or CORS setup is needed. The point table stays on the server.

## 2. Download the project and data into week12

Download [census-full-stack.zip](downloads/census-full-stack.zip) and extract the `census-full-stack` project folder **inside `week12`**. Download the two source files into **`week12/data`** using the links in Part A before continuing. The ZIP contains application code and a dependency reference, not the source datasets.

```text
week12/
├── data/
│   ├── census_sample.parquet
│   └── districts_2015.geojson
├── lecture-12C.ipynb
└── census-full-stack/
    ├── environment.yml
    ├── prepare_data.py
    ├── database.py
    ├── app.py
    ├── test_app.py
    └── static/
```

Run shell commands below in a **terminal inside `week12/census-full-stack`**, rather than a notebook code cell. The notebook is a walkthrough; starting a long-running server in a cell would block Run All.

## 3. Add server tools to your existing geospatial environment

Keep using **`geospatial`**, the same environment as earlier in the course and Part A. In a terminal, install the new tools into it:

```bash
conda activate geospatial
conda install -c conda-forge fastapi uvicorn httpx
```

FastAPI defines the routes and validation. Uvicorn runs the HTTP server. HTTPX is used by the optional notebook client and integration tests. If you skipped Part A, also run `conda install -c conda-forge python-duckdb=1.5.6 pyarrow`. Keep this environment activated for every command below.

Select **Python (geospatial)** for notebook cells. Restart the kernel after installation. If this kernel is missing, follow Part A's kernel registration steps. The included `environment.yml` is a dependency reference; no new environment is needed.

DuckDB and its Spatial extension perform the queries. Leaflet runs in the browser and requires no Python package installation. No OpenAI API key or additional Census API request is involved.

## 4. Prepare a separate application database

From inside the extracted project directory:

```bash
python prepare_data.py --data ../data
```

The first preparation installs DuckDB Spatial and needs internet access. It creates **`data/census.duckdb`**, separate from 12A's database, using:

1. `ST_Read` to select 2015 Pennsylvania district polygons.
2. `read_parquet` plus `ST_Covers` to select the Census dots inside those polygons.
3. `ST_Transform` to store both geometries in **EPSG:26918**, with meter units.
4. A `point_district` table assigning every selected dot once; boundary ties use the smallest matching district ID, as in 12A.

The checked teaching files produce **63,552 sample dots across 18 historical districts**. Your result should match when using the same inputs. The script refuses to overwrite an existing demo database. Stop the server before removing this demo database and rebuilding it. Source files stay unchanged.

## 5. Start Python and open the application

```bash
python -m uvicorn app:app --host 127.0.0.1 --port 8000
```

Keep this terminal running. Open:

- [The web map](http://127.0.0.1:8000/).
- [Interactive FastAPI documentation](http://127.0.0.1:8000/docs).
- [Health check](http://127.0.0.1:8000/api/health).

Press **Ctrl+C** to stop the server. Localhost is your computer; a classmate cannot open your localhost URL. Do not use `python -m http.server` for this project: that serves files but does not execute FastAPI routes. Leaflet's CDN files and OpenStreetMap basemap tiles need internet access.

**Try the app:** click **Try Philadelphia City Hall**, change the radius from 2 km to 5 km, select a category, and click another location inside Pennsylvania. The district shading responds to the category selector, while the count panel responds to the location, radius, and category. Click outside Pennsylvania to see the validation message.

## 6. Inspect the API contract

| Request | Response | Frontend use |
| --- | --- | --- |
| `/api/health` | Status and total sample dots | Check preparation/server readiness |
| `/api/districts?category=all` | GeoJSON polygons with `district_id` and `sample_dots` | Draw district shading and tooltips |
| `/api/nearby?lon=-75.1636&lat=39.9526&radius_m=2000&category=all` | Total and category counts around a location | Update the count panel |

Open the browser's developer tools → **Network**, click the map, and inspect the `/api/nearby` URL and JSON response. In `/docs`, expand `GET /api/nearby`, choose **Try it out**, enter the City Hall values above, and execute it. The valid category values are `all`, `w`, `b`, `a`, `h`, and `o`; the allowed radius is 100–20,000 m. Invalid parameter types/ranges return HTTP 422; a location outside the selected district coverage returns HTTP 400.

Optional Python client: keep the server running in its terminal and run the following cell with the `geospatial` notebook kernel. It makes a real HTTP request; it does not open the database directly.

In [1]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/api/nearby",
    params={"lon": -75.1636, "lat": 39.9526, "radius_m": 2000, "category": "all"},
    timeout=30,
)
response.raise_for_status()
result = response.json()
print("Sample dots within 2 km:", result["sample_dots"])
result["categories"]

Sample dots within 2 km: 532


[{'category': 'w', 'label': 'White', 'sample_dots': 341}, {'category': 'b', 'label': 'Black', 'sample_dots': 94}, {'category': 'a', 'label': 'Asian', 'sample_dots': 57}, {'category': 'h', 'label': 'Hispanic', 'sample_dots': 32}, {'category': 'o', 'label': 'Other', 'sample_dots': 8}]

## 7. Trace the Python backend

Open [app.py](examples/census-full-stack/app.py). `@app.get('/api/nearby')` registers a URL handler. FastAPI checks typed query parameters before calling it. Each request opens its own **read-only connection** and closes it after the query. The `lifespan` startup check detects a missing database before the app accepts requests.

The core query is:

```sql
WITH location AS (
    SELECT ST_Transform(ST_Point(?, ?), 'EPSG:4326', 'EPSG:26918',
                        always_xy := true) AS geom
)
SELECT p.category, count(*)
FROM census_points p, location q
WHERE ST_DWithin(p.geom, q.geom, ?)
  AND (? = 'all' OR p.category = ?)
GROUP BY p.category
ORDER BY p.category
```

Python passes `[lon, lat, radius_m, category, category]` separately from the SQL text. The browser supplies allowed parameter values, never SQL. Distances use projected meter coordinates, not degrees. The server returns only grouped counts; district geometry is transformed back to EPSG:4326 for GeoJSON.

A radius near the Pennsylvania border counts only dots in the Pennsylvania extract. The web-map circle is illustrative: the analytical distance comes from the projected SQL query, not a screen measurement.

## 8. Trace the frontend

Open [static/app.js](examples/census-full-stack/static/app.js). Its `queryLocation()` function builds the request using `URLSearchParams`, calls `fetch()`, checks the response, and updates the page. A request counter discards older responses if several queries overlap. Errors are shown in a live status region.

```javascript
const params = new URLSearchParams({
  lon: latlng.lng, lat: latlng.lat,
  radius_m: radius.value, category: category.value
});
const response = await fetch(`/api/nearby?${params}`);
const data = await response.json();
```

Notice the ordering: Leaflet exposes `lat`/`lng`, while the API names X as `lon` and Y as `lat`. The frontend draws historical polygons with `L.geoJSON()` and displays a shared legend for district counts in the selected category. The scale is recomputed when that category changes; compare numbers rather than interpreting colors across different filters. These are counts, not area-normalized density.

## 9. Test the system

With the environment activated, run from the project directory after preparing the database:

```bash
python -m unittest -v
```

The included tests compare API district totals against direct SQL for all categories, check radius expansion and category filtering, reject invalid inputs, and verify frontend files and health. They use FastAPI's in-process test client; the Uvicorn server need not be running.

Then check the real browser flow: map loads; the City Hall button returns a result; category/radius changes update the response; an outside-state click gives a clear message; the last-response link shows the actual JSON. Use Network to distinguish a failed API request from a missing basemap tile.

## 10. Explain hosting: static files versus Python

Quarto can publish this lecture as static HTML, but it does not keep the API running. GitHub Pages can serve a map frontend and exported JSON, but cannot run this Python backend.

For an application deployment, use a **Python-capable host** and keep this demo's frontend/API on that same host:

1. Install the additional packages in `geospatial` and supply the two existing source files during preparation.
2. Build the database once; retain it on persistent storage or rebuild it during deployment.
3. Configure a start command equivalent to `python -m uvicorn app:app --host 0.0.0.0 --port $PORT`, using the host's supplied port.
4. Enable the host's HTTPS endpoint and check `/api/health`, `/`, and `/docs`.

This lecture runs locally and does not create a hosting account or publish the app. Use one server worker for the teaching demo. Many concurrent writers and shared editing would be a separate PostgreSQL/PostGIS design exercise; this app exposes read-only analytical queries.

To render this notebook with saved outputs:

```bash
quarto render lecture-12C.ipynb --to html --no-execute
```

The runnable server stays a separate project; rendering cannot turn localhost links into a public API.

## Exercises and troubleshooting

1. Explain the request path for a map click, naming the frontend, backend, and database responsibilities.
2. Add a 3 km choice to the radius menu without changing the server's validation limits.
3. Query two locations with the same radius/category; explain what you can and cannot infer from the sample counts.
4. Add a backend summary endpoint and inspect it in `/docs` before connecting a new frontend control.

| Symptom | Check |
| --- | --- |
| Missing `fastapi`, `uvicorn`, or `httpx` | Activate `geospatial` and run `conda install -c conda-forge fastapi uvicorn httpx` |
| Database missing at startup | Run `prepare_data.py --data ...` first, in the project directory |
| Cannot find Parquet/GeoJSON | Point `--data` at the local `week12/data` folder containing both downloaded files |
| Spatial installation fails | Check internet access and the DuckDB version from the environment file |
| Port 8000 is in use | Stop the previous server or choose `--port 8001` and use that URL everywhere |
| HTML opens but API fails | Use Uvicorn's URL rather than `file://` or a plain static server |
| Map is blank but API works | Check CDN/OSM requests and internet access in browser Network |
| Rebuild refused | Stop the API and remove only the demo's `data/census.duckdb` before rebuilding |

**Data provenance:** [Week 11 sample preparation](../week11-large-data-viz-lidar/data/samples/provenance.json). Historical source categories are limited representations of identity; sample counts are not population estimates, density, or current demographics.

**References:** [FastAPI static files](https://fastapi.tiangolo.com/tutorial/static-files/) · [FastAPI testing](https://fastapi.tiangolo.com/tutorial/testing/) · [DuckDB spatial functions](https://duckdb.org/docs/stable/core_extensions/spatial/functions.html).